# 6.6 - Video & 3D Generation

**Module 6 - Image & Multimodal Generation** | Netsetos GenAI Engineering

This notebook works through Video & 3D Generation hands-on: Feel the data cost of video; Compress first, then patch: latent video diffusion; Generate video via an API: submit, then poll; 3D from photos: what one Gaussian holds; Text/image-to-3D: a game-ready mesh in seconds; Choose a model by the constraint that binds.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

A near-empty prep cell: an optional `torch` install line for fresh/Colab environments and a note that the lesson uses seeded random tensors. Nothing here runs a model - it just makes sure PyTorch is importable before the cells that build tensors.

> **Optional install** - uncomment the `pip install torch` line if you're on Colab or a clean machine.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install torch -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

Environment prep only. The single dependency for the whole notebook is PyTorch, used to build small random tensors that stand in for images and video; the API/3D cells further down import their own SDKs at the point of use rather than here.

**How the code works - step by step**
- **Commented `!pip install torch -q`** - uncomment to install PyTorch in a fresh environment; already present in most local setups.
- **Reproducibility note** - flags that the lesson leans on seeded random values, so tensor shapes and value counts are stable across runs.

**In one line:** make sure `torch` is available, then move on.

**What you'll see:** (no output - environment prep)

## 1 - Feel the data cost of video

Before any architecture, get a gut feel for *why* video is hard: it's just images stacked on a time axis, but there are a lot of them. This cell counts the raw values in one frame versus one second versus ten seconds of video at the same resolution.

In [ ]:
# A video is just images stacked on a time axis. Let's feel the data cost.
import torch

image    = torch.randn(3, 512, 512)          # one RGB frame: (channels, H, W)
video_1s = torch.randn(24, 3, 512, 512)      # 1 second at 24 fps: (frames, C, H, W)
video_10s = torch.randn(240, 3, 512, 512)    # 10 seconds: 240 frames

def millions(t):
    return t.numel() / 1e6              # total values, in millions

print(f"1 image:      {millions(image):.2f} M values")
print(f"1 sec video:  {millions(video_1s):.2f} M values  ({video_1s.shape[0]}x an image)")
print(f"10 sec video: {millions(video_10s):.2f} M values  ({video_10s.shape[0]}x an image)")
# Output: 1 image:      0.79 M values
# Output: 1 sec video:  18.87 M values  (24x an image)
# Output: 10 sec video: 188.74 M values  (240x an image)

**Explanation**

A measurement harness, not a model call. It builds three random tensors - a single frame, one second, and ten seconds of 24 fps video - and counts total elements to quantify the blow-up. The point is the 240x multiplier: that single number is what makes video expensive and clips short.

**How the code works - step by step**
- **`image`** - a `(3, 512, 512)` tensor: one RGB frame, channels-height-width.
- **`video_1s` / `video_10s`** - the same frame shape with a **frames** axis prepended: `(24, 3, 512, 512)` and `(240, 3, 512, 512)`, i.e. 1s and 10s at 24 fps.
- **`millions(t)`** - divides `t.numel()` (total values in the tensor) by 1e6 so the counts are readable.
- **Three prints** - report each tensor's size in millions of values and its multiple of a single image.

**In one line:** add a frames axis, count the values, and watch a 10s clip balloon to 240x one image.

**What you'll see:** Three lines: 1 image ~0.79 M values, 1 sec video ~18.87 M (24x an image), 10 sec video ~188.74 M (240x an image) - the concrete cost of adding a time axis.

## 2 - Compress first, then patch: latent video diffusion

The 240x cost from Step 1 is tamed by two moves: a 3D VAE compresses the clip into a small latent, then a Diffusion Transformer cuts that latent into 3D "spacetime patches" it can denoise. This cell puts numbers on both moves.

In [ ]:
import torch

video_10s = torch.randn(240, 3, 512, 512)   # raw: (frames, C, H, W)

# A 3D VAE compresses ~8x per spatial side AND groups frames in time.
latent = torch.randn(60, 4, 64, 64)         # (time-groups, ch, H/8, W/8)

# A Diffusion Transformer cuts the LATENT into spacetime patches -
# like ViT patches from 6.4, but 3D (a bit of space across a few frames).
patch_t, patch_s = 2, 2                      # 2 time-groups x 2x2 spatial per patch
n_patches = (60 // patch_t) * (64 // patch_s) * (64 // patch_s)

print(f"raw values:        {video_10s.numel():,}")
print(f"latent values:     {latent.numel():,}  ({video_10s.numel() // latent.numel()}x smaller)")
print(f"spacetime patches: {n_patches:,}  <- the tokens the transformer denoises")
# Output: raw values:        188,743,680
# Output: latent values:     983,040  (192x smaller)
# Output: spacetime patches: 30,720  <- the tokens the transformer denoises

**Explanation**

Another sizing calculation, this time of the winning architecture rather than the raw cost. It compares the raw clip to a compressed latent, then computes how many spacetime-patch tokens the transformer actually operates on. The takeaway is that compression plus patching shrinks the problem by orders of magnitude - it's Stable Diffusion's latent trick and the ViT's patch trick, extended one axis into time.

**How the code works - step by step**
- **`video_10s`** - the raw `(240, 3, 512, 512)` clip from Step 1.
- **`latent`** - a stand-in `(60, 4, 64, 64)` tensor: the 3D VAE compresses ~8x per spatial side (512->64) and groups frames in time (240->60).
- **`patch_t, patch_s = 2, 2`** - patch size: 2 time-groups by a 2x2 spatial tile per token.
- **`n_patches`** - `(60//2) * (64//2) * (64//2)` = the count of 3D spacetime patches the transformer treats as tokens.
- **Three prints** - raw value count, latent value count (with the compression ratio), and the number of spacetime patches actually denoised.

**In one line:** compress the clip to a latent (~192x smaller), then slice it into 3D patch-tokens for the transformer.

**What you'll see:** raw values 188,743,680; latent values 983,040 (192x smaller); spacetime patches 30,720 - the tokens the Diffusion Transformer denoises instead of 188M raw values.

## 3 - Generate video via an API: submit, then poll

Video generation is a job you submit and wait on, not a call that returns the MP4 immediately. This cell shows the async shape with a Runway image-to-video request: create a task, hold its id, poll until it reaches a terminal status.

> **Needs a Runway API key** (set `RUNWAYML_API_SECRET`) - illustrative; won't run without credentials and a real image URL.

In [ ]:
# pip install runwayml   -  image-to-video via a hosted API (async: submit, then poll)
import os, time
from runwayml import RunwayML

client = RunwayML(api_key=os.environ["RUNWAYML_API_SECRET"])

# 1) SUBMIT a job - you get a task id back, NOT the video.
task = client.image_to_video.create(
    model="gen4_turbo",
    prompt_image="https://example.com/product.jpg",   # a still to animate
    prompt_text="Camera slowly orbits the product, soft studio light, smooth motion.",
    duration=5,                                         # seconds
    ratio="1280:720",
)

# 2) POLL until it is ready (in production, prefer a webhook over a loop).
while True:
    task = client.tasks.retrieve(task.id)
    if task.status in ("SUCCEEDED", "FAILED"):
        break
    time.sleep(5)

print(task.status)
print(task.output[0] if task.status == "SUCCEEDED" else task.failure)
# Output: SUCCEEDED
# Output: https://runway-outputs.../gen4_turbo_orbit.mp4

**Explanation**

A realistic API-client cell demonstrating the universal submit-then-poll pattern that every hosted video API shares. `create()` hands back a task id instantly; a loop calls `retrieve()` until the status is `SUCCEEDED` or `FAILED`. Read it as the shape of the interaction, not a synchronous "give me the video now" call - that would time out.

**How the code works - step by step**
- **`client = RunwayML(...)`** - authenticates using the key from the environment.
- **`client.image_to_video.create(...)`** - **submits** the job (model `gen4_turbo`, a still `prompt_image` to animate, a `prompt_text` steering camera/motion, 5s duration, 1280:720) and returns a task with an id - not the video.
- **`while True: ... retrieve(task.id)`** - **polls** every 5 seconds, breaking once the status is terminal; a webhook is preferred in production over a sleep loop.
- **Final prints** - the status, then `task.output[0]` (a signed, time-limited result URL) on success or `task.failure` on failure.

**In one line:** submit a job -> hold the id -> poll until done -> get a temporary result URL.

**What you'll see:** Prints `SUCCEEDED` then a signed MP4 URL (e.g. a gen4_turbo orbit clip). The URL expires, so in practice you download and re-host the bytes rather than persist the vendor link.

## 4 - 3D from photos: what one Gaussian holds

Shift from time to space. Gaussian Splatting turns dozens of photos into a scene you can fly through, built from millions of tiny coloured blobs. This cell defines the data held by a *single* one of those blobs and sketches the capture-to-render pipeline in comments.

In [ ]:
# What ONE 3D Gaussian holds - a scene is millions of these.
from dataclasses import dataclass

@dataclass
class Gaussian3D:
    position: tuple   # (x, y, z) - where the blob sits in space
    scale:    tuple   # (sx, sy, sz) - its size and shape (the covariance)
    color:    tuple   # (r, g, b) - view-dependent, stored as spherical harmonics
    opacity:  float   # 0..1 - how solid vs see-through it is

blob = Gaussian3D(position=(0.0, 0.1, 2.3), scale=(0.02, 0.02, 0.05),
                  color=(0.8, 0.2, 0.2), opacity=0.9)
print(blob)
print("a real indoor room ~ 1-2M Gaussians, rendered at 100+ FPS")
# Output: Gaussian3D(position=(0.0, 0.1, 2.3), scale=(0.02, 0.02, 0.05), color=(0.8, 0.2, 0.2), opacity=0.9)
# Output: a real indoor room ~ 1-2M Gaussians, rendered at 100+ FPS

# Run it on your own photos with nerfstudio (open, well-documented):
#   pip install nerfstudio
#   ns-process-data images --data ./photos --output-dir ./scene   # SfM camera poses (COLMAP)
#   ns-train splatfacto --data ./scene                            # optimize the Gaussians (minutes on a GPU)
#   ns-viewer --load-config ./outputs/.../config.yml              # fly through the result
# Prefer a phone app? Luma or Polycam do capture -> splat in the cloud.

**Explanation**

A reference/data-shape cell, not a training run. A small dataclass makes concrete exactly what one 3D Gaussian stores - position, scale, colour, opacity - so "millions of explicit blobs" stops being abstract. The commented `nerfstudio` commands show the real pipeline you'd run on your own photos.

**How the code works - step by step**
- **`Gaussian3D` dataclass** - four fields: `position` (x,y,z location), `scale` (size/shape, the covariance), `color` (r,g,b, stored as spherical harmonics so it varies with viewing angle), and `opacity` (0..1, solid vs see-through).
- **`blob = Gaussian3D(...)`** - one example blob, printed to show a filled-in instance.
- **Two prints** - the blob's fields and a scale note (a real indoor room is ~1-2M Gaussians rendered at 100+ FPS).
- **Commented `nerfstudio` commands** - the actual pipeline: `ns-process-data` (SfM/COLMAP camera poses) -> `ns-train splatfacto` (optimize the Gaussians) -> `ns-viewer` (fly through); Luma/Polycam do this as phone apps.

**In one line:** one Gaussian = position + size + colour + opacity; a scene is millions of them, optimized to match your photos.

**What you'll see:** Prints the `Gaussian3D(...)` instance with its four fields, then the note that a real room is ~1-2M Gaussians at 100+ FPS. No optimization happens here - it's a data-structure illustration.

## 5 - Text/image-to-3D: a game-ready mesh in seconds

The reliable 2026 path to a 3D asset is text -> one clean 2D image -> feed-forward image-to-3D. This cell shows that second half with Hunyuan3D: a reference image goes in, an untextured mesh comes out, then a paint pass adds PBR texture and the result is exported as glTF.

> **Needs the Hunyuan3D models + a GPU** - illustrative; downloads large open checkpoints and expects a reference image file.

In [ ]:
# The reliable 2026 pipeline: text -> one clean 2D image -> feed-forward image-to-3D.
# Step A: make/choose a single clean reference image (SD/Flux from Lesson 6.2).
# Step B: convert it to 3D with an open feed-forward model (seconds, not hours).
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline   # shape (open)
from hy3dgen.texgen import Hunyuan3DPaintPipeline                # texture (open)
from PIL import Image

img = Image.open("chair_reference.png")
shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2")
mesh = shape(image=img)[0]                 # image -> untextured mesh (shape only)
paint = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")
mesh = paint(mesh, image=img)              # add PBR texture from the reference image
mesh.export("chair.glb")                   # glTF: import into Blender / Unity / Unreal
print("saved chair.glb  -  expect 15-30 min of cleanup: topology, UVs, the occluded back")
# Output: saved chair.glb  -  expect 15-30 min of cleanup: topology, UVs, the occluded back

**Explanation**

An API-usage cell for an open feed-forward image-to-3D pipeline. Two models run in sequence - one produces the shape, one paints it - because image-to-3D (a clean reference pins down the object) is more reliable than asking for 3D straight from text. Feed-forward means seconds per asset, unlike SDS optimization which takes hours.

**How the code works - step by step**
- **`Image.open("chair_reference.png")`** - loads the single clean reference (made with SD/Flux from Lesson 6.2).
- **`Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(...)`** then **`shape(image=img)[0]`** - converts the image into an untextured mesh (shape only).
- **`Hunyuan3DPaintPipeline.from_pretrained(...)`** then **`paint(mesh, image=img)`** - adds PBR texture derived from the reference.
- **`mesh.export("chair.glb")`** - writes a glTF file importable into Blender/Unity/Unreal.
- **Print** - flags realistic expectations: 15-30 min of cleanup on topology, UVs, and the occluded back.

**In one line:** reference image -> mesh -> paint -> export `.glb`, then budget artist cleanup.

**What you'll see:** Prints `saved chair.glb - expect 15-30 min of cleanup: topology, UVs, the occluded back` - the honest reminder that feed-forward output is a strong start, not a finished asset.

## 6 - Choose a model by the constraint that binds

The 2026 landscape splits into closed leaders and open self-host options, and the right pick depends on what constrains you - not the top benchmark. This cell encodes that as a tiny lookup from a named need to a recommended model.

In [ ]:
# Pick a video model by the constraint that binds - not by benchmark alone.
def choose_video_model(need: str) -> str:
    table = {
        "top_quality":    "Veo 3.1 (native audio, up to 4K) or Runway Gen-4.5 (best control)",
        "cheap_at_scale": "Kling 3.0 (roughly $0.50/clip, fast, multilingual lip-sync)",
        "privacy":       "self-host Wan (Apache-2.0, 1.3B runs in ~8 GB VRAM)",
        "native_audio":  "Veo 3.1 native synchronized audio; Kling 3.0 lip-sync",
    }
    return table.get(need, "start hosted; move to open self-host when cost or privacy forces it")

for need in ["top_quality", "cheap_at_scale", "privacy", "native_audio"]:
    print(f"{need:14} -> {choose_video_model(need)}")
# Output: top_quality    -> Veo 3.1 (native audio, up to 4K) or Runway Gen-4.5 (best control)
# Output: cheap_at_scale -> Kling 3.0 (roughly $0.50/clip, fast, multilingual lip-sync)
# Output: privacy        -> self-host Wan (Apache-2.0, 1.3B runs in ~8 GB VRAM)
# Output: native_audio   -> Veo 3.1 native synchronized audio; Kling 3.0 lip-sync

**Explanation**

A decision-procedure cell, pure Python with no model call. It maps four common binding constraints - top quality, cheap at scale, privacy, native audio - to a concrete recommendation, with a sensible default for anything else. The message is that cost-per-output and privacy decide as much as quality, the lesson Sora taught by shutting down despite its capability.

**How the code works - step by step**
- **`choose_video_model(need)`** - a dict lookup keyed by constraint: `top_quality` -> Veo 3.1 / Runway Gen-4.5, `cheap_at_scale` -> Kling 3.0, `privacy` -> self-hosted Wan, `native_audio` -> Veo 3.1 / Kling.
- **`table.get(need, ...)`** - falls back to "start hosted, move to open self-host when cost or privacy forces it" for unknown needs.
- **`for need in [...]`** - runs all four constraints through the function and prints each mapping.

**In one line:** name the constraint that binds, get the model that fits it.

**What you'll see:** Four aligned lines mapping each need to its recommendation (top_quality -> Veo 3.1/Runway Gen-4.5, cheap_at_scale -> Kling 3.0, privacy -> self-host Wan, native_audio -> Veo 3.1/Kling) - a decision table, and a reminder that prices and version names are mid-2026 and move monthly.

Across six cells you saw one machine reused: video is diffusion (6.1) plus the ViT patch idea (6.4) with a time axis, made affordable by a 3D VAE and kept consistent by temporal attention; 3D-from-photos is millions of explicit Gaussian blobs optimized to your shots; 3D-from-a-prompt is feed-forward image-to-3D with an artist-cleanup budget. The through-line is cost: a 10-second clip is ~240x an image, which is why clips are short, APIs are async submit-then-poll, and model choice is driven by the constraint that binds (quality, price at scale, privacy, native audio) rather than by benchmarks alone. Next, Lesson 6.7 closes the module with the last modality - voice - and Module 15 develops the provenance and deepfake responsibilities this lesson only names.